# 🚀 Crypto Sentiment Analysis with Groq (Production-Ready Notebook)

In [ ]:
# Install dependencies (run once)
# !pip install groq pandas pydantic python-dotenv

In [ ]:
import os
import json
import pandas as pd
from groq import Groq
from pydantic import BaseModel, ValidationError
from dotenv import load_dotenv

In [ ]:
load_dotenv()
client = Groq(api_key=os.getenv('GROQ_API_KEY'))

## Structured Output

In [ ]:
class SentimentResponse(BaseModel):
    sentiment: str
    confidence: float
    reasoning: str
    market_insight: str
    recommended_action: str

## Guardrails

In [ ]:
def validate_input(text):
    if not isinstance(text, str):
        raise ValueError('Invalid input')
    if len(text.strip()) < 5:
        raise ValueError('Too short')
    if 'ignore previous' in text.lower():
        raise ValueError('Injection detected')
    return text.strip()

## Memory

In [ ]:
class Memory:
    def __init__(self):
        self.history = []
    def add(self, i, o):
        self.history.append({'input': i, 'output': o})
    def get(self):
        return self.history[-5:]
memory = Memory()

## LLM Analysis

In [ ]:
def analyze(text, context=None):
    prompt = f'''You are a crypto analyst.
Return JSON:
{{
 "sentiment": "", "confidence": 0.0, "reasoning": "", "market_insight": "", "recommended_action": ""
}}
Text: {text}
Context: {context}
'''
    res = client.chat.completions.create(
        model='llama3-70b-8192',
        messages=[{'role':'user','content':prompt}],
        temperature=0.2
    )
    raw = res.choices[0].message.content
    try:
        data = json.loads(raw)
        return SentimentResponse(**data).dict()
    except Exception as e:
        return {'error': str(e), 'raw': raw}

## Sample Data

In [ ]:
df = pd.DataFrame({'text':[
    'Bitcoin is pumping hard!',
    'Market crash incoming',
    'ETH is stable',
    'Huge bullish breakout',
    'Crypto is dead'
]})
df

## Run Pipeline

In [ ]:
results = []
for t in df['text']:
    try:
        clean = validate_input(t)
        ctx = memory.get()
        r = analyze(clean, ctx)
        memory.add(clean, r)
        results.append(r)
    except Exception as e:
        results.append({'error': str(e)})
res_df = pd.DataFrame(results)
res_df

In [ ]:
res_df.to_csv('results.csv', index=False)
print('Saved results.csv')